# Crear la clase  Target

Hay que crear el archivo competencia_01.csv, usando el competencia_01_crudo.csv con una variable adicional llamada clase_ternaria, que contenga las categorías **CONTINUA, BAJA+1 y BAJA+2**

Para esto primero es necesario descargar el dataset.

In [1]:
from pathlib import Path
import requests

file_url = "https://storage.googleapis.com/open-courses/dmeyf2026-9c6f/competencia_01_crudo.csv"
dataset_filename = "competencia_01_crudo.csv"

current_dir = Path.cwd()
if (current_dir / ".git").exists():
    project_root = current_dir
elif (current_dir.parent / ".git").exists():
    project_root = current_dir.parent
else:
    raise RuntimeError("No se encontro la raiz del repositorio dmeyf2026")

dataset_path = project_root / "data"
dataset_path.mkdir(parents=True, exist_ok=True)
file_path = dataset_path / dataset_filename
output_file = dataset_path / "competencia_01.csv"
dataset_file_sql = file_path.as_posix()
output_file_sql = output_file.as_posix()

if file_path.exists():
    print("Ya esta el dataset, continue")
else:
    print("Falta el dataset, a bajarlo...")
    try:
        response = requests.get(file_url, stream=True)
        response.raise_for_status()

        with file_path.open("wb") as f:
            for chunk in response.iter_content(chunk_size=8192):
                f.write(chunk)
        print(f"Dataset descargado en {file_path}")
    except requests.exceptions.RequestException as e:
        print(f"Error descargando el archivo: {e}")
    except Exception as e:
        print(f"Ocurrio un error inesperado: {e}")


Ya esta el dataset, continue


Luego, para generar la clase vamos a usar el lenguaje **SQL**, a través de una base de datos **OLAP** llamada **DuckDB**.

La documentación la puede encontrar [aquí](https://duckdb.org/docs/current/) Procedemos a instalar componentes adicionales para utilizar directamente el lenguaje en la notebook

## Dependencias locales

Las dependencias se administran mediante `requirements.txt` dentro de un ambiente virtual. Desde la raíz del repositorio, ejecutar `python -m pip install -r requirements.txt`.

Configuramos el entorno de ejecución. Si ya tiene todo instalado, solo necesita ejecutar esta celda para empezar a usar **duckdb**

In [2]:
import duckdb
import pandas as pd

%load_ext sql
%config SqlMagic.autopandas = True
%config SqlMagic.feedback = False
%config SqlMagic.displaycon = False

%sql duckdb:///

Support for third party widgets will remain active for the duration of the session. To disable support:

Y ya podemos usar **SQL** dentro de una notebook!

In [3]:
%%sql
select 'hola mundo'

,'hola mundo'
0,hola mundo


Cargamos el archivo `.csv` a una tabla pasando su ubicación

In [4]:
%%sql
create or replace table competencia_01_crudo as
select
    *
from read_csv_auto('{{dataset_file_sql}}')

,Success


Hagamos unas queries básicas para comprobar que todo esta funcionando bien.



In [5]:
%sql select * from competencia_01_crudo limit 5

,numero_de_cliente,foto_mes,active_quarter,cliente_vip,internet,cliente_edad,cliente_antiguedad,mrentabilidad,mrentabilidad_annual,mcomisiones,...,Visa_madelantodolares,Visa_fultimo_cierre,Visa_mpagado,Visa_mpagospesos,Visa_mpagosdolares,Visa_fechaalta,Visa_mconsumototal,Visa_cconsumos,Visa_cadelantosefectivo,Visa_mpagominimo
0,12159854,202103,1,0,0,56,134,688.41,26701.84,98.82,...,0.0,1,0.0,-16247.77,0.0,4056,15732.34,1,0,1137.81
1,12159858,202103,1,0,0,48,102,78.43,24418.75,-73.62,...,NaN,<NA>,NaN,NaN,NaN,<NA>,NaN,<NA>,<NA>,NaN
2,12160484,202103,1,0,0,60,55,8101.55,3162.23,13399.50,...,0.0,1,0.0,-31103.23,0.0,1632,2860.54,2,0,19858.89
3,12160591,202103,1,0,0,46,275,14825.78,138050.05,1146.27,...,0.0,1,0.0,-13733.44,0.0,2122,1419.36,3,0,1231.65
4,12160747,202103,1,0,0,47,194,2015.61,31240.49,1791.25,...,0.0,1,0.0,0.00,0.0,5901,1286.93,1,0,82.11


In [6]:
%%sql
select
    foto_mes
    , count(*) as cantidad -- cuenta cuantos casos hay en cada foto_mes
                           -- y lo guarda en un campo llamado cantidad
from competencia_01_crudo
group by foto_mes

,foto_mes,cantidad
0,202103,162900
1,202104,163284
2,202105,163768
3,202106,164114
4,202107,164348
5,202108,164647


Antes de avanzar con la construcción del **target** veamos repasemos un poco de **SQL**

## WITH

Un escenario común es necesitar realizar una consulta a no a una tabla, sino a otra consulta. Una forma tradicional de hacerlo es usando es:
* Creando una tabla temporal con esa consulta
* Usando `selects from select`.

Esta necesidad es muy común, pero como todo en la vida hay peros.
- Crear una tabla en una base de datos que va a ser usada UNA SOLA VEZ ensucia los esquemas con un montón de tablas que no son útiles y además del desorden ocupan espacio innecesario
- El `select from select`, debe ser de las cosas peor vistas por los administradores de BBDD. Tienen sus razones técnicas, en general la recomendación es no usarlo a menos que uno conozca bien como funciona la base de datos por dentro, ya que hay casos donde terminan ocasionando muchos problemas

Y que hacer? bueno hay una clausula en **SQL** llamada `WITH`, que nos permite crear esas (plural) tablas para ser usadas en la **query** principal. Ejemplo si queremos comparar el valor de una variable con su promedio a través de todos los meses.


In [7]:
%%sql
with promedios as (
    select
        numero_de_cliente
        , avg(mrentabilidad) as avg_mrentabilidad
    from competencia_01_crudo
    group by numero_de_cliente
) select
    cp.numero_de_cliente
    , cp.foto_mes
    , if(cp.mrentabilidad > p.avg_mrentabilidad, 'mayor', 'menor') as comparacion
from competencia_01_crudo cp
join promedios p using (numero_de_cliente)


,numero_de_cliente,foto_mes,comparacion
0,30429417,202104,menor
1,30429506,202104,mayor
2,30429784,202104,menor
3,30429948,202104,menor
4,30429976,202104,menor
...,...,...,...
983056,78249586,202108,menor
983057,78249850,202108,menor
983058,78250419,202108,menor
983059,78253625,202108,menor


- Usar esa clausula tiene muchos beneficios, entre ellos sumar legibilidad de código y le permite al motor de **SQL** optimizar mejor, ya que entiende todo lo que se buscar hacer y planifica como resolverlo en el menor tiempo posible.

**NOTA**: Hay un tipo de tabla que se llaman temporales, que se borran solas una vez que uno se desconecta de la base de datos. Es una mejor alternativa frente al `CREATE`, sin embargo si la tabla que se esta creando se usa una sola vez es conveniente usar el `WITH` porque permite optimizar la ejecución. Si esa tabla va a ser usada mas de una vez dentro del proceso, las tablas temporales son la mejor opción.

# Funciones Analíticas

Las funciones analíticas en SQL son un conjunto de funciones que te permiten realizar cálculos avanzados sobre un conjunto de filas relacionadas dentro de una consulta conservando las filas individuales y calculando sobre ventanas de datos relacionadas.

Veamos un ejemplo para que quede más claro:

In [8]:
%%sql
select
    numero_de_cliente
    , foto_mes
    , mrentabilidad
from competencia_01_crudo
where numero_de_cliente = 12159854
order by foto_mes

,numero_de_cliente,foto_mes,mrentabilidad
0,12159854,202103,688.41
1,12159854,202104,1622.57
2,12159854,202105,1987.98
3,12159854,202106,2580.75
4,12159854,202107,2686.85
5,12159854,202108,2707.62


In [9]:
%%sql
select
    numero_de_cliente
    , foto_mes
    , mrentabilidad
    , lead(mrentabilidad, 1) over (partition by numero_de_cliente order by foto_mes) as mrentabilidad_mas_1
    , lag(mrentabilidad, 1) over (partition by numero_de_cliente order by foto_mes) as mrentabilidad_menos_1
    , lead(mrentabilidad, 2) over (partition by numero_de_cliente order by foto_mes) as mrentabilidad_mas_2
    , lag(mrentabilidad, 2) over (partition by numero_de_cliente order by foto_mes) as mrentabilidad_menos_2
from competencia_01_crudo
where numero_de_cliente = 12159854
order by foto_mes



,numero_de_cliente,foto_mes,mrentabilidad,mrentabilidad_mas_1,mrentabilidad_menos_1,mrentabilidad_mas_2,mrentabilidad_menos_2
0,12159854,202103,688.41,1622.57,NaN,1987.98,NaN
1,12159854,202104,1622.57,1987.98,688.41,2580.75,NaN
2,12159854,202105,1987.98,2580.75,1622.57,2686.85,688.41
3,12159854,202106,2580.75,2686.85,1987.98,2707.62,1622.57
4,12159854,202107,2686.85,2707.62,2580.75,NaN,1987.98
5,12159854,202108,2707.62,NaN,2686.85,NaN,2580.75


* Qué paso?
* ¿Cómo esta construyendo las nuevas variables?
* ¿ Nos puede ayudar a construir targets ?

¿Dondé puedo leer un poco más de esta magia negra? https://duckdb.org/docs/sql/window_functions.html

## Los casos raros

Que pasa con el cliente **60112254** y la foto **202104**?


In [10]:
%%sql
select
    numero_de_cliente
    , foto_mes
from competencia_01_crudo
where numero_de_cliente = 60112254
order by foto_mes

,numero_de_cliente,foto_mes
0,60112254,202103
1,60112254,202105
2,60112254,202106
3,60112254,202107
4,60112254,202108


* ¿Cuál debería ser la clase para cada periodo?

### Una pista

Podemos generar todas las posibles combinaciones de clientes y periodos de manera muy simple

In [11]:
%%sql
with periodos as (
  select distinct foto_mes from competencia_01_crudo
), clientes as (
  select distinct numero_de_cliente from competencia_01_crudo
)
select numero_de_cliente, foto_mes from clientes cross join periodos
where numero_de_cliente = 60112254
order by foto_mes

,numero_de_cliente,foto_mes
0,60112254,202103
1,60112254,202104
2,60112254,202105
3,60112254,202106
4,60112254,202107
5,60112254,202108


Solo nos queda saber si estuvo o no el banco el cliente en ese periodo y armar el

In [12]:
%%sql
with periodos as (
    select distinct foto_mes from competencia_01_crudo -- Esto también se puede hacer con secuencias
), clientes as (
    select distinct numero_de_cliente from competencia_01_crudo
), todo as (
    select numero_de_cliente, foto_mes from clientes cross join periodos
)
select
    t.numero_de_cliente
    , t.foto_mes
    , if(c.numero_de_cliente is null, 0, 1) as mes_0
from todo t
left join competencia_01_crudo c using (numero_de_cliente, foto_mes)
where t.numero_de_cliente = 60112254
order by foto_mes

,numero_de_cliente,foto_mes,mes_0
0,60112254,202103,1
1,60112254,202104,0
2,60112254,202105,1
3,60112254,202106,1
4,60112254,202107,1
5,60112254,202108,1


Antes de continuar, tomémonos un momento para reflexionar. Con todas estas piezas, ¿cómo podemos ensamblar el target?

Una vez que lo tengamos claro sobre el papel, proceda a completar el código que sigue.

In [13]:
%%sql
create or replace table competencia_01 as
with periodos as (
    select distinct foto_mes from competencia_01_crudo
), clientes as (
    select distinct numero_de_cliente from competencia_01_crudo
), todo as (
    select numero_de_cliente, foto_mes from clientes cross join periodos
), clase_ternaria as (
    select
        c.*
        , if(c.numero_de_cliente is null, 0, 1) as mes_0
        , lead(mes_0, 1) over (partition by t.numero_de_cliente order by foto_mes) as mes_1
        , lead(mes_0, 2) over (partition by t.numero_de_cliente order by foto_mes) as mes_2
        , case
            when mes_1 = 0 then 'BAJA+1'
            when mes_1 = 1 and mes_2 = 0 then 'BAJA+2'
            when mes_1 = 1 and mes_2 = 1 then 'CONTINUA'
            else null
          end as clase_ternaria
    from todo t
    left join competencia_01_crudo c using (numero_de_cliente, foto_mes)
) select
  * EXCLUDE (mes_0, mes_1, mes_2)
from clase_ternaria
where mes_0 = 1

,Success


## Validación independiente

Se reconstruye el target mediante joins a uno y dos meses calendario y se compara fila por fila con la solución basada en `LEAD`.

In [14]:
%%sql
with base as (
    select
        numero_de_cliente,
        foto_mes,
        strptime(cast(foto_mes as varchar), '%Y%m') as mes
    from competencia_01_crudo
),
alternativa as (
    select
        t.numero_de_cliente,
        t.foto_mes,
        case
            -- Agosto no tiene ningún mes futuro disponible.
            when t.foto_mes = 202108 then null

            -- Si falta el mes siguiente, es BAJA+1.
            when t1.numero_de_cliente is null then 'BAJA+1'

            -- En julio, si llegó a agosto, falta septiembre
            -- para distinguir BAJA+2 de CONTINUA.
            when t.foto_mes = 202107 then null

            -- Si estuvo el mes siguiente pero falta dos meses
            -- después, es BAJA+2.
            when t2.numero_de_cliente is null then 'BAJA+2'

            else 'CONTINUA'
        end as clase_alternativa
    from base t
    left join base t1
        on t1.numero_de_cliente = t.numero_de_cliente
       and t1.mes = t.mes + interval 1 month
    left join base t2
        on t2.numero_de_cliente = t.numero_de_cliente
       and t2.mes = t.mes + interval 2 month
)
select
    count(*) as filas_comparadas,
    count(*) filter (
        where c.clase_ternaria
              is distinct from a.clase_alternativa
    ) as diferencias
from competencia_01 c
join alternativa a
    using (numero_de_cliente, foto_mes)

,filas_comparadas,diferencias
0,983061,0


Revisamos que todo salga como esperamos

In [15]:
%sql select count(*) from competencia_01

,count_star()
0,983061


Y vemos la cardinalidad de las clases por periodo

* ¿Cuál es la nominalidad de cada clase?


In [16]:
%%sql
PIVOT competencia_01
on clase_ternaria
USING count(numero_de_cliente)
GROUP BY foto_mes

,foto_mes,BAJA+1,BAJA+2,CONTINUA
0,202103,1019,960,160921
1,202104,964,1139,161181
2,202105,1143,870,161755
3,202106,874,1098,162142
4,202107,1103,0,0
5,202108,0,0,0


## Proporción del target

Se calcula la proporción de cada clase solamente para los meses con horizonte completo (`202103` a `202106`).

In [17]:
%%sql
select
    foto_mes,
    clase_ternaria,
    count(*) as cantidad,
    round(
        100.0 * count(*) /
        sum(count(*)) over (partition by foto_mes),
        4
    ) as porcentaje
from competencia_01
where foto_mes between 202103 and 202106
group by foto_mes, clase_ternaria
order by foto_mes, clase_ternaria

,foto_mes,clase_ternaria,cantidad,porcentaje
0,202103,BAJA+1,1019,0.6255
1,202103,BAJA+2,960,0.5893
2,202103,CONTINUA,160921,98.7851
3,202104,BAJA+1,964,0.5904
4,202104,BAJA+2,1139,0.6976
5,202104,CONTINUA,161181,98.7121
6,202105,BAJA+1,1143,0.6979
7,202105,BAJA+2,870,0.5312
8,202105,CONTINUA,161755,98.7708
9,202106,BAJA+1,874,0.5326


## Guardar tabla en `.csv`

La tabla final se exporta a la ruta portátil definida al comienzo de la notebook.

In [18]:
%%sql
COPY competencia_01 TO '{{output_file_sql}}' (FORMAT CSV, HEADER)

,Success



## El pato del amor

DuckDB no es una base de datos SQL estándar; su sintaxis moderna facilita significativamente la escritura de consultas. A continuación, te menciono algunas características destacadas:

+ Uso de variables: En DuckDB, no es necesario reescribir las variables; puedes utilizar tus variables una vez definidas, lo que simplifica el código.

+ Función IF: DuckDB soporta la función IF(condición, valor_si_verdadero, valor_si_falso), lo que hace que tu código sea más claro y legible en comparación con el uso de CASE WHEN.

+ Exclusión de campos: Puedes utilizar SELECT * EXCLUDE(field1, ...) para excluir campos específicos de la lista de selección, lo que ahorra tiempo y esfuerzo.

+ Alias en cláusulas: Es posible utilizar alias en las cláusulas WHERE, GROUP BY y HAVING, lo que añade flexibilidad a la escritura de consultas.

+ Uso de alias en cálculos: DuckDB permite el uso de alias dentro del cálculo de otros campos, facilitando la manipulación y transformación de datos en una consulta.

Estas son solo algunas de las capacidades que hacen de DuckDB una herramienta poderosa. Te recomiendo leer la documentación y el blog oficial para explorar más a fondo su potencial.